# Анализ CRM по сегментам бизнеса

Файл на сервере: `crm_last_version.csv`

**Сегментация по `X_AREA_RESP`:**

| X_AREA_RESP | Сегмент |
|-------------|--------|
| ДМСБ (микро) | МкБ |
| ДМСБ (малый) | МСБ |
| ДМСБ (средний) | МСБ |
| ДКБ | ККБ |

**Период:** 2023-2025 включительно  
**Дедупликация:** уникальные комбинации `X_INN` + `NUMBER_FP_SFP` + `IDENTIFICATION_DATE`

In [ ]:
import pandas as pd       # библиотека для работы с таблицами (DataFrame)
import numpy as np        # библиотека для числовых операций (используется pandas под капотом)
import matplotlib.pyplot as plt  # библиотека для построения графиков

import warnings
# Подавляем предупреждения (FutureWarning, DeprecationWarning и т.д.),
# чтобы вывод ячеек был чистым — только наши print/display.
# Код работает и без этой строки, но в выводе будут жёлтые предупреждения.
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", None)  # отображаем полные наименования в таблицах

# ── Пути к файлам ──
DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"  # папка с исходными данными
CRM_FILE = f"{DATA_DIR}/crm_last_version.csv"             # выгрузка CRM с факторами проблемности
REF_FILE = f"{DATA_DIR}/ref_book.csv"                      # справочник ФП/СФП (номера, наименования, ЗО)

# ── Период анализа ──
DATE_FROM = pd.Timestamp("2023-01-01")  # начало периода (включительно)
DATE_TO   = pd.Timestamp("2025-12-31")  # конец периода (включительно)

# ── Маппинг X_AREA_RESP → укрупнённый сегмент бизнеса ──
# В CRM у каждой записи есть колонка X_AREA_RESP (зона ответственности).
# Мы объединяем 4 значения в 3 сегмента для анализа:
#   МкБ  = микробизнес
#   МСБ  = малый + средний бизнес (объединяем два подразделения ДМСБ)
#   ККБ  = крупный корпоративный бизнес
SEGMENT_MAP = {
    "ДМСБ (микро)":   "МкБ",
    "ДМСБ (малый)":   "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ":            "ККБ",
}

# ── Маппинг X_AREA_RESP → ЗО для ФП/СФП (как в справочнике ref_book.csv) ──
# В справочнике колонка "ЗО для ФП/СФП" называется иначе, чем X_AREA_RESP в CRM.
# Этот маппинг нужен, чтобы при джойне CRM со справочником
# сопоставить правильные наименования факторов.
# Пример: в CRM "ДМСБ (микро)", а в справочнике та же ЗО называется "ДМ (микро)".
ZO_MAP = {
    "ДМСБ (микро)":   "ДМ (микро)",
    "ДМСБ (малый)":   "ДМСБ (малый)",
    "ДМСБ (средний)": "ДМСБ (средний)",
    "ДКБ":            "ДКБ",
}
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]


def normalize_inn(s):
    """
    Приводит ИНН к единому строковому формату.

    Зачем это нужно:
    - Excel может сохранить ИНН как число, добавив ".0" (например "7707083893.0")
    - В разных выгрузках ИНН может быть с ведущими нулями ("007707083893") или без
    - Без нормализации pandas считает "7707083893", "7707083893.0" и "007707083893"
      разными значениями, хотя это один и тот же ИНН

    Логика:
    1. Убираем ".0" с конца (артефакт Excel)
    2. Убираем ведущие нули
    3. Дополняем нулями слева до 10 знаков (юрлицо) или 12 знаков (ИП)
    """
    if pd.isna(s):           # если значение пустое (NaN) — возвращаем None
        return None
    s = str(s).strip()       # приводим к строке, убираем пробелы по краям
    if s.endswith(".0"):     # "7707083893.0" → "7707083893"
        s = s[:-2]
    s = s.lstrip("0") or "0"  # "007707083893" → "7707083893"; сохраняем "0" если всё нули
    if len(s) <= 10:
        return s.zfill(10)   # дополняем до 10 знаков: "7707083893" → "7707083893"
    return s.zfill(12)       # дополняем до 12 знаков (для ИП с 12-значным ИНН)


print("Параметры заданы.")

## 1. Загрузка и подготовка данных

In [ ]:
# ── Загрузка CSV-файла ──
# sep=";"          — разделитель колонок (в российских выгрузках обычно точка с запятой)
# dtype=str — читаем все колонки как строки, чтобы избежать потери ведущих нулей в ИНН
# dtype=str        — читаем ВСЕ колонки как строки, чтобы pandas не превращал
#                    ИНН/КПП/ОГРН в числа (и не терял ведущие нули)
# low_memory=False — отключаем экономию памяти, чтобы pandas не угадывал типы по частям
df = pd.read_csv(CRM_FILE, dtype=str, low_memory=False)
print(f"Загружено: {len(df):,} строк")

# ── Нормализация ИНН ──
# Применяем функцию normalize_inn к каждому значению колонки X_INN,
# результат записываем в новую колонку "inn"
df["inn"] = df["X_INN"].apply(normalize_inn)

# ── Преобразование даты ──
# Колонка IDENTIFICATION_DATE хранится как строка (например "15.03.2024").
# Преобразуем в формат даты pandas (Timestamp), чтобы можно было фильтровать по периоду.
# dayfirst=True — указываем, что формат ДД.ММ.ГГГГ (а не ММ/ДД/ГГГГ как в США)
# errors="coerce" — если строку не удаётся распарсить, ставим NaT (а не падаем с ошибкой)
df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")

# ── Фильтр по периоду ──
# Оставляем только строки, где дата идентификации попадает в наш период 2023-2025.
# .copy() — создаём независимую копию DataFrame, чтобы дальнейшие изменения
# не вызывали предупреждений pandas (SettingWithCopyWarning)
df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()
print(f"После фильтра {DATE_FROM:%d.%m.%Y} - {DATE_TO:%d.%m.%Y}: {len(df):,} строк")

# ── Фильтр по источникам (VAL) ──
if "VAL" in df.columns:
    before_val = len(df)
    df = df[df["VAL"].str.strip().isin(ALLOWED_SOURCES)].copy()
    print(f"Фильтр по источникам ({', '.join(ALLOWED_SOURCES)}): "
          f"{len(df):,} строк (удалено {before_val - len(df):,})")

# ── Переименование TYPE_FP ──
# В реальной выгрузке crm_last_version значения в колонке TYPE_FP записаны
# латиницей: "FP" и "SFP". Переименовываем в кириллицу для единообразия.
# Для синтетических данных (где уже "ФП"/"СФП") replace ничего не изменит.
df["TYPE_FP"] = df["TYPE_FP"].replace({"FP": "ФП", "SFP": "СФП"})

# ── Маппинг сегментов ──
# Приводим X_AREA_RESP к строке и убираем пробелы по краям (на случай "ДКБ " с пробелом)
df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
# Создаём новую колонку "segment", подставляя значение из словаря SEGMENT_MAP.
# Если X_AREA_RESP не найден в словаре — segment будет NaN.
df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

# Проверяем, есть ли значения X_AREA_RESP, которые не попали в маппинг (неизвестные ЗО).
# Если есть — выводим их для диагностики, затем исключаем из анализа.
unmapped = df[df["segment"].isna()]["X_AREA_RESP"].value_counts()
if len(unmapped) > 0:
    print(f"\nНемаппированные значения X_AREA_RESP (будут исключены):")
    print(unmapped.to_string())

# Оставляем только строки с известным сегментом
df = df[df["segment"].notna()].copy()
print(f"После маппинга сегментов: {len(df):,} строк")

# ── Нормализация номера фактора ──
df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()

# ── Удаление строк без номера фактора / ИНН ──
before_drop = len(df)
df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()
print(f"После dropna(inn, NUMBER_FP_SFP): {len(df):,} строк (удалено {before_drop - len(df):,})")

# ── Дедупликация ──
before_dedup = len(df)
df = df.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()
print(f"После дедупликации (INN + fp_num + TYPE_FP + DATE): {len(df):,} строк (удалено {before_dedup - len(df):,} дублей)")

# ── Колонка для группировки по месяцам ──
# Преобразуем дату в «период» (год-месяц), например 2024-03.
# Это нужно для построения помесячной динамики в секциях 6 и 7.
df["year_month"] = df["dt"].dt.to_period("M")

## 2. Проверка: ИНН в нескольких сегментах

In [ ]:
# ── Проверка: может ли один ИНН оказаться в нескольких сегментах? ──
# Такое возможно, если компания проходила по разным подразделениям
# (например, сначала в МкБ, потом перешла в МСБ).
# Это важно знать, потому что при подсчёте «уникальных ИНН по сегментам»
# такие ИНН будут посчитаны в каждом сегменте, и сумма превысит общее число.

# Для каждого ИНН считаем, сколько РАЗНЫХ сегментов у него есть
seg_per_inn = df.groupby("inn")["segment"].nunique()

# Отбираем ИНН, у которых больше одного сегмента
multi_seg = seg_per_inn[seg_per_inn > 1]

print("=" * 60)
print("ПРОВЕРКА: ИНН в нескольких сегментах")
print("=" * 60)
print(f"  Всего уникальных ИНН:              {len(seg_per_inn):,}")
print(f"  ИНН в одном сегменте:              {(seg_per_inn == 1).sum():,}")
print(f"  ИНН в нескольких сегментах:        {len(multi_seg):,}")

if len(multi_seg) > 0:
    # Выводим детали по каждому «мультисегментному» ИНН (до 15 штук)
    print(f"\n  Детали (до 15 примеров):")
    print("  " + "-" * 56)
    for inn in multi_seg.index[:15]:
        subset = df[df["inn"] == inn]             # все записи этого ИНН
        segs = subset["segment"].unique()         # в каких сегментах он встречается
        areas = subset["X_AREA_RESP"].unique()    # какие оригинальные значения X_AREA_RESP
        n = len(subset)                           # сколько всего записей
        print(f"  ИНН {inn}  ->  {n} записей")
        print(f"    X_AREA_RESP: {', '.join(areas)}")
        print(f"    Сегменты:    {', '.join(segs)}")
else:
    print("\n  Каждый ИНН принадлежит ровно одному сегменту.")

## 3. Уникальные ИНН по сегментам

In [ ]:
# ── Подсчёт уникальных ИНН по сегментам ──

# Общее число уникальных ИНН во всём DataFrame (без разделения на сегменты)
total_inn = df["inn"].nunique()

# Порядок сегментов для отображения (от малого к крупному)
seg_order = ["МкБ", "МСБ", "ККБ"]

# Для каждого сегмента считаем количество уникальных ИНН.
# .reindex(seg_order) — гарантирует порядок + подставляет 0, если сегмент пуст.
inn_by_seg = df.groupby("segment")["inn"].nunique().reindex(seg_order, fill_value=0)

print("=" * 50)
print("УНИКАЛЬНЫЕ ИНН")
print("=" * 50)
print(f"  Всего:  {total_inn:,}")
print(f"  " + "-" * 30)
for seg in seg_order:
    cnt = inn_by_seg.get(seg, 0)
    pct = cnt / total_inn * 100 if total_inn else 0  # доля от общего числа
    print(f"  {seg:<6s} {cnt:>8,}  ({pct:.1f}%)")

# Дублируем в виде таблицы для наглядности
tbl = pd.DataFrame({
    "Сегмент": seg_order,
    "Уникальных ИНН": [inn_by_seg.get(s, 0) for s in seg_order],
})
tbl.loc[len(tbl)] = ["ИТОГО", total_inn]  # строка-итого внизу
display(tbl.style.hide(axis="index"))      # отображаем без стандартного числового индекса

## 4. Общая статистика по ФП/СФП

In [ ]:
# Сколько всего уникальных номеров факторов встречается в данных за период
total_factors = df["NUMBER_FP_SFP"].nunique()

# Из них — сколько уникальных номеров встречаются как ФП
fp_factors = df[df["TYPE_FP"] == "ФП"]["NUMBER_FP_SFP"].nunique()

# И сколько — как СФП
sfp_factors = df[df["TYPE_FP"] == "СФП"]["NUMBER_FP_SFP"].nunique()

# Общее количество записей (событий)
total_events = len(df)
fp_events = (df["TYPE_FP"] == "ФП").sum()
sfp_events = (df["TYPE_FP"] == "СФП").sum()

print("=" * 60)
print("ОБЩАЯ СТАТИСТИКА ПО ФП/СФП")
print("=" * 60)
print(f"  Всего записей (событий):              {total_events:,}")
print(f"    из них ФП:                           {fp_events:,}")
print(f"    из них СФП:                          {sfp_events:,}")
print()
print(f"  Уникальных номеров факторов:           {total_factors}")
print(f"    встречаются как ФП:                  {fp_factors}")
print(f"    встречаются как СФП:                 {sfp_factors}")

In [ ]:
# Проверка: какие номера факторов встречаются и как ФП, и как СФП
fp_set = set(df[df["TYPE_FP"] == "ФП"]["NUMBER_FP_SFP"].dropna().unique())
sfp_set = set(df[df["TYPE_FP"] == "СФП"]["NUMBER_FP_SFP"].dropna().unique())
both = sorted(fp_set & sfp_set)

print("=" * 70)
print("ПРОВЕРКА: факторы, встречающиеся и как ФП, и как СФП")
print("=" * 70)
print(f"  Уникальных номеров только ФП:    {len(fp_set - sfp_set)}")
print(f"  Уникальных номеров только СФП:   {len(sfp_set - fp_set)}")
print(f"  Встречаются в обоих типах:       {len(both)}")
print(f"  Итого уникальных:                {len(fp_set | sfp_set)}")
print(f"  ({len(fp_set)} ФП + {len(sfp_set)} СФП - {len(both)} пересечение = {len(fp_set) + len(sfp_set) - len(both)})")
print()

if both:
    _ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
    _ref.columns = _ref.columns.str.strip()
    _name_map = _ref.drop_duplicates(subset=["№"]).set_index("№")["Наименование"].to_dict()

    print(f"  Номера факторов в пересечении ({len(both)} шт.):")
    print("  " + "-" * 66)
    for num in both:
        fp_cnt = (df[(df["NUMBER_FP_SFP"] == num) & (df["TYPE_FP"] == "ФП")]).shape[0]
        sfp_cnt = (df[(df["NUMBER_FP_SFP"] == num) & (df["TYPE_FP"] == "СФП")]).shape[0]
        name = _name_map.get(num, "— наименование не найдено в справочнике —")
        print(f"    {num:<10s}  ФП: {fp_cnt:>6,}  |  СФП: {sfp_cnt:>5,}")
        print(f"              {name}")
        print()
else:
    print("  Пересечений нет — все номера принадлежат только одному типу.")

## 5. Количество ФП/СФП по сегментам и среднее на ИНН

In [ ]:
# ── Кросс-таблица: количество записей по (сегмент × тип фактора) ──
fp_stats = df.groupby(["segment", "TYPE_FP"]).size().unstack(fill_value=0)
fp_stats = fp_stats.reindex(seg_order, fill_value=0)

for col in ["ФП", "СФП"]:
    if col not in fp_stats.columns:
        fp_stats[col] = 0

fp_stats = fp_stats[["ФП", "СФП"]]
fp_stats["Всего"] = fp_stats["ФП"] + fp_stats["СФП"]

print("=" * 60)
print("КОЛИЧЕСТВО ФП/СФП ПО СЕГМЕНТАМ")
print("=" * 60)

totals = fp_stats.sum()
fp_stats.loc["ИТОГО"] = totals
display(fp_stats)

# ── Среднее количество ФП/СФП на один ИНН по сегментам ──
# Для каждого сегмента: общее число событий / число уникальных ИНН
print()
print("=" * 60)
print("СРЕДНЕЕ КОЛИЧЕСТВО ФП/СФП НА ОДИН ИНН")
print("=" * 60)

avg_rows = []
for seg in seg_order:
    seg_df = df[df["segment"] == seg]
    n_inn = seg_df["inn"].nunique()
    if n_inn == 0:
        continue
    n_fp = (seg_df["TYPE_FP"] == "ФП").sum()
    n_sfp = (seg_df["TYPE_FP"] == "СФП").sum()
    n_total = len(seg_df)
    avg_rows.append({
        "Сегмент": seg,
        "Уник. ИНН": n_inn,
        "ФП": n_fp,
        "СФП": n_sfp,
        "Всего": n_total,
        "Ср. ФП/ИНН": round(n_fp / n_inn, 2),
        "Ср. СФП/ИНН": round(n_sfp / n_inn, 2),
        "Ср. всего/ИНН": round(n_total / n_inn, 2),
    })

# Строка ИТОГО
all_inn = df["inn"].nunique()
all_fp = (df["TYPE_FP"] == "ФП").sum()
all_sfp = (df["TYPE_FP"] == "СФП").sum()
all_total = len(df)
avg_rows.append({
    "Сегмент": "ИТОГО",
    "Уник. ИНН": all_inn,
    "ФП": all_fp,
    "СФП": all_sfp,
    "Всего": all_total,
    "Ср. ФП/ИНН": round(all_fp / all_inn, 2) if all_inn else 0,
    "Ср. СФП/ИНН": round(all_sfp / all_inn, 2) if all_inn else 0,
    "Ср. всего/ИНН": round(all_total / all_inn, 2) if all_inn else 0,
})

df_avg = pd.DataFrame(avg_rows)
display(df_avg.style.hide(axis="index"))

# ── Столбчатая диаграмма ──
fp_plot = fp_stats.drop("ИТОГО")
ax = fp_plot[["ФП", "СФП"]].plot(kind="bar", figsize=(8, 4), edgecolor="white")
ax.set_title("Количество ФП и СФП по сегментам", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Количество")
ax.set_xticklabels(seg_order, rotation=0)
for c in ax.containers:
    ax.bar_label(c, fmt="%d", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Топ-20 самых частых ФП

In [ ]:
# Считаем частоту каждого номера фактора среди записей с TYPE_FP = "ФП"
fp_top20 = (
    df[df["TYPE_FP"] == "ФП"]
    .groupby("NUMBER_FP_SFP").size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
fp_top20.columns = ["№ ФП", "Кол-во"]
total_fp = (df["TYPE_FP"] == "ФП").sum()
fp_top20["% от всех ФП"] = (fp_top20["Кол-во"] / total_fp * 100).round(2).astype(str) + "%"
fp_top20.index = range(1, len(fp_top20) + 1)

print("=" * 50)
print(f"ТОП-20 САМЫХ ЧАСТЫХ ФП  (всего ФП: {total_fp:,})")
print("=" * 50)
display(fp_top20)

## 7. Топ-20 самых частых СФП

In [ ]:
# Аналогично секции 6, но для СФП
sfp_top20 = (
    df[df["TYPE_FP"] == "СФП"]
    .groupby("NUMBER_FP_SFP").size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)
sfp_top20.columns = ["№ СФП", "Кол-во"]
total_sfp = (df["TYPE_FP"] == "СФП").sum()
sfp_top20["% от всех СФП"] = (sfp_top20["Кол-во"] / total_sfp * 100).round(2).astype(str) + "%"
sfp_top20.index = range(1, len(sfp_top20) + 1)

print("=" * 50)
print(f"ТОП-20 САМЫХ ЧАСТЫХ СФП  (всего СФП: {total_sfp:,})")
print("=" * 50)
display(sfp_top20)

## 8. Топ-20 самых частых ФП по сегментам

In [ ]:
# Для каждого сегмента выводим топ-20 ФП по частоте
for seg in seg_order:
    seg_fp = df[(df["segment"] == seg) & (df["TYPE_FP"] == "ФП")]
    top = (
        seg_fp.groupby("NUMBER_FP_SFP").size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top.columns = ["№ ФП", "Кол-во"]
    seg_total = len(seg_fp)
    top["% от ФП сегмента"] = (top["Кол-во"] / seg_total * 100).round(2).astype(str) + "%"
    top.index = range(1, len(top) + 1)

    print("=" * 55)
    print(f"  {seg}  —  Топ-20 ФП  (всего ФП в сегменте: {seg_total:,})")
    print("=" * 55)
    display(top)
    print()

## 9. Топ-20 самых частых СФП по сегментам

In [ ]:
# Аналогично секции 8, но для СФП
for seg in seg_order:
    seg_sfp = df[(df["segment"] == seg) & (df["TYPE_FP"] == "СФП")]
    top = (
        seg_sfp.groupby("NUMBER_FP_SFP").size()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top.columns = ["№ СФП", "Кол-во"]
    seg_total = len(seg_sfp)
    top["% от СФП сегмента"] = (top["Кол-во"] / seg_total * 100).round(2).astype(str) + "%" if seg_total > 0 else "—"
    top.index = range(1, len(top) + 1)

    print("=" * 55)
    print(f"  {seg}  —  Топ-20 СФП  (всего СФП в сегменте: {seg_total:,})")
    print("=" * 55)
    display(top)
    print()

## 10. Подключение справочника ФП/СФП (ref_book.csv)

In [ ]:
# ── Загрузка справочника ──
df_ref = pd.read_csv(REF_FILE, sep="\t", encoding="utf-16", dtype=str)
df_ref.columns = df_ref.columns.str.strip()

# ── Маппинг X_AREA_RESP → ЗО (как в справочнике) ──
# Нужен для джойна: в CRM "ДМСБ (микро)", в справочнике "ДМ (микро)"
df["zo"] = df["X_AREA_RESP"].map(ZO_MAP)

# ── Джойн по двум ключам: номер фактора + ЗО ──
# Один и тот же номер (например "35.2") в разных ЗО имеет РАЗНОЕ наименование,
# поэтому джойн по двум ключам — обязателен.
df_merged = df.merge(
    df_ref[["№", "ЗО для ФП/СФП", "Наименование"]],
    left_on=["NUMBER_FP_SFP", "zo"],
    right_on=["№", "ЗО для ФП/СФП"],
    how="left",
)

no_match = df_merged["Наименование"].isna().sum()
total_rows = len(df_merged)
print(f"Джойн со справочником: {total_rows:,} строк, не найдено наименование: {no_match:,} ({no_match/total_rows*100:.1f}%)")


def build_top(data, n=5):
    """
    Топ-N номеров ФП/СФП по частоте с подтягиванием наименований.
    Считаем по NUMBER_FP_SFP (не по паре номер+наименование),
    потому что в МСБ один номер может иметь разные наименования из разных ЗО.
    """
    counts = data.groupby("NUMBER_FP_SFP").size().nlargest(n)
    rows = []
    for num, cnt in counts.items():
        names = data.loc[data["NUMBER_FP_SFP"] == num, "Наименование"].dropna().unique()
        rows.append({
            "№ ФП/СФП": num,
            "Наименование": " / ".join(sorted(names)) if len(names) > 0 else "",
            "Кол-во": cnt,
        })
    result = pd.DataFrame(rows)
    result.index = range(1, len(result) + 1)
    return result

## 11. Топ-5 самых частых ФП по сегментам (с наименованиями)

In [ ]:
full_fp_names = {}
for seg in seg_order:
    seg_data = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "ФП")]
    if len(seg_data) == 0:
        print(f"\n  {seg}: нет записей ФП")
        continue

    full_fp_names[seg] = build_top(seg_data, len(seg_data))
    zo_vals = sorted(seg_data["zo"].dropna().unique())
    zo_label = ", ".join(zo_vals)
    print("=" * 80)
    print(f"  {seg}  (ЗО: {zo_label})  —  Топ-5 самых частых ФП")
    print("=" * 80)
    display(build_top(seg_data, 5))
    print()

## 12. Топ-5 самых частых СФП по сегментам (с наименованиями)

In [ ]:
full_sfp_names = {}
for seg in seg_order:
    seg_data = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "СФП")]
    if len(seg_data) == 0:
        print(f"\n  {seg}: нет записей СФП")
        continue

    full_sfp_names[seg] = build_top(seg_data, len(seg_data))
    zo_vals = sorted(seg_data["zo"].dropna().unique())
    zo_label = ", ".join(zo_vals)
    print("=" * 80)
    print(f"  {seg}  (ЗО: {zo_label})  —  Топ-5 самых частых СФП")
    print("=" * 80)
    display(build_top(seg_data, 5))
    print()

## 13. Самые частые комбинации из 2 ФП по сегментам

Для каждого ИНН берём все уникальные номера ФП, генерируем все пары, считаем какие пары встречаются чаще всего.

In [ ]:
from itertools import combinations

combo_fp_by_seg = {}
for seg in seg_order:
    seg_fp = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "ФП")]

    inn_factors = seg_fp.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))

    pair_counts = {}
    for factors in inn_factors:
        if len(factors) >= 2:
            for pair in combinations(factors, 2):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    if not pair_counts:
        print(f"\n  {seg}: нет ИНН с 2+ различными ФП\n")
        continue

    all_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
    rows_all = []
    for (f1, f2), cnt in all_pairs:
        n1 = seg_fp.loc[seg_fp["NUMBER_FP_SFP"] == f1, "Наименование"].dropna().unique()
        n2 = seg_fp.loc[seg_fp["NUMBER_FP_SFP"] == f2, "Наименование"].dropna().unique()
        rows_all.append({
            "ФП 1": f1, "Наименование 1": " / ".join(sorted(n1)) if len(n1) else "",
            "ФП 2": f2, "Наименование 2": " / ".join(sorted(n2)) if len(n2) else "",
            "Кол-во ИНН": cnt,
        })
    full_result = pd.DataFrame(rows_all)
    full_result.index = range(1, len(full_result) + 1)
    combo_fp_by_seg[seg] = full_result

    zo_vals = sorted(seg_fp["zo"].dropna().unique())
    print("=" * 95)
    print(f"  {seg}  (ЗО: {', '.join(zo_vals)})  —  Топ комбинаций из 2 ФП")
    print("=" * 95)
    display(full_result.head(10))
    print()

## 14. Самые частые комбинации из 2 СФП по сегментам

In [ ]:
combo_sfp_by_seg = {}
for seg in seg_order:
    seg_sfp = df_merged[(df_merged["segment"] == seg) & (df_merged["TYPE_FP"] == "СФП")]

    inn_factors = seg_sfp.groupby("inn")["NUMBER_FP_SFP"].apply(lambda x: sorted(x.dropna().unique()))

    pair_counts = {}
    for factors in inn_factors:
        if len(factors) >= 2:
            for pair in combinations(factors, 2):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

    if not pair_counts:
        print(f"\n  {seg}: нет ИНН с 2+ различными СФП\n")
        continue

    all_pairs = sorted(pair_counts.items(), key=lambda x: -x[1])
    rows_all = []
    for (f1, f2), cnt in all_pairs:
        n1 = seg_sfp.loc[seg_sfp["NUMBER_FP_SFP"] == f1, "Наименование"].dropna().unique()
        n2 = seg_sfp.loc[seg_sfp["NUMBER_FP_SFP"] == f2, "Наименование"].dropna().unique()
        rows_all.append({
            "СФП 1": f1, "Наименование 1": " / ".join(sorted(n1)) if len(n1) else "",
            "СФП 2": f2, "Наименование 2": " / ".join(sorted(n2)) if len(n2) else "",
            "Кол-во ИНН": cnt,
        })
    full_result = pd.DataFrame(rows_all)
    full_result.index = range(1, len(full_result) + 1)
    combo_sfp_by_seg[seg] = full_result

    zo_vals = sorted(seg_sfp["zo"].dropna().unique())
    print("=" * 95)
    print(f"  {seg}  (ЗО: {', '.join(zo_vals)})  —  Топ комбинаций из 2 СФП")
    print("=" * 95)
    display(full_result.head(10))
    print()

## 15. Динамика ФП по месяцам (по сегментам)

In [ ]:
# ── Оставляем только строки с TYPE_FP = "ФП" (факторы проблемности) ──
# СФП (существенные факторы проблемности) анализируются отдельно в секции 7
df_fp = df[df["TYPE_FP"] == "ФП"].copy()

# ── Сводная таблица: кол-во ФП по (месяц × сегмент) ──
# groupby по year_month и segment, считаем количество строк,
# .unstack() разворачивает сегменты в колонки.
# Результат — таблица, где строки = месяцы, колонки = сегменты, значения = кол-во ФП.
pivot_fp = df_fp.groupby(["year_month", "segment"]).size().unstack(fill_value=0)

# Гарантируем порядок колонок (МкБ, МСБ, ККБ) и заполняем нулями пустые
pivot_fp = pivot_fp.reindex(columns=seg_order, fill_value=0)

# Преобразуем индекс из Period ("2024-03") в строку для корректного отображения на графике
pivot_fp.index = pivot_fp.index.astype(str)

# Таблица с точными значениями по месяцам
tbl_fp = pivot_fp.copy()
tbl_fp["Итого"] = tbl_fp[seg_order].sum(axis=1)
tbl_fp.index.name = "Месяц"

print("=" * 60)
print("ДИНАМИКА ФП ПО МЕСЯЦАМ (таблица)")
print("=" * 60)
display(tbl_fp)

# ── Линейный график ──
fig, ax = plt.subplots(figsize=(14, 5))           # создаём фигуру 14×5 дюймов
pivot_fp.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)  # линия с точками для каждого сегмента
ax.set_title("Количество выявленных ФП по месяцам (2023-2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество ФП")
ax.legend(title="Сегмент")               # легенда с заголовком «Сегмент»
ax.tick_params(axis="x", rotation=45)     # поворачиваем подписи месяцев на 45°, чтобы не слипались
plt.tight_layout()                        # автоподбор отступов
plt.show()

## 16. Динамика СФП по месяцам (по сегментам)

In [ ]:
# ── Аналогично секции 6, но для СФП (существенные факторы проблемности) ──
df_sfp = df[df["TYPE_FP"] == "СФП"].copy()

# Сводная таблица: кол-во СФП по (месяц × сегмент)
pivot_sfp = df_sfp.groupby(["year_month", "segment"]).size().unstack(fill_value=0)
pivot_sfp = pivot_sfp.reindex(columns=seg_order, fill_value=0)
pivot_sfp.index = pivot_sfp.index.astype(str)

# Таблица с точными значениями по месяцам
tbl_sfp = pivot_sfp.copy()
tbl_sfp["Итого"] = tbl_sfp[seg_order].sum(axis=1)
tbl_sfp.index.name = "Месяц"

print("=" * 60)
print("ДИНАМИКА СФП ПО МЕСЯЦАМ (таблица)")
print("=" * 60)
display(tbl_sfp)

# ── Линейный график ──
fig, ax = plt.subplots(figsize=(14, 5))
pivot_sfp.plot(ax=ax, marker="o", markersize=3, linewidth=1.2)
ax.set_title("Количество выявленных СФП по месяцам (2023-2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("Количество СФП")
ax.legend(title="Сегмент")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
_t = lambda s: s[:31]
excel_path = f'{DATA_DIR}/Приложение к отчету "Анализ динамики ФП по сегментам бизнеса".xlsx'

with pd.ExcelWriter(excel_path, engine="openpyxl") as w:
    tbl.to_excel(w, sheet_name="Уникальные ИНН", index=False)

    fp_stats.to_excel(w, sheet_name="ФП-СФП по сегментам")
    df_avg.to_excel(w, sheet_name="Среднее ФП на ИНН", index=False)

    fp_top20.to_excel(w, sheet_name="Топ-20 ФП (общие)")
    sfp_top20.to_excel(w, sheet_name="Топ-20 СФП (общие)")

    for seg in seg_order:
        seg_fp_data = df[df["segment"] == seg]
        fp_t = seg_fp_data[seg_fp_data["TYPE_FP"] == "ФП"].groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).reset_index()
        fp_t.columns = ["№ ФП", "Кол-во"]
        fp_t.to_excel(w, sheet_name=_t(f"ФП ({seg})"), index=False)

        sfp_t = seg_fp_data[seg_fp_data["TYPE_FP"] == "СФП"].groupby("NUMBER_FP_SFP").size().sort_values(ascending=False).reset_index()
        sfp_t.columns = ["№ СФП", "Кол-во"]
        sfp_t.to_excel(w, sheet_name=_t(f"СФП ({seg})"), index=False)

    for seg, df_fn in full_fp_names.items():
        df_fn.to_excel(w, sheet_name=_t(f"ФП с назв. ({seg})"), index=False)
    for seg, df_fn in full_sfp_names.items():
        df_fn.to_excel(w, sheet_name=_t(f"СФП с назв. ({seg})"), index=False)

    for seg, df_c in combo_fp_by_seg.items():
        df_c.to_excel(w, sheet_name=_t(f"Комбинации ФП ({seg})"), index=False)
    for seg, df_c in combo_sfp_by_seg.items():
        df_c.to_excel(w, sheet_name=_t(f"Комбинации СФП ({seg})"), index=False)

    tbl_fp.to_excel(w, sheet_name="Динамика ФП по месяцам")
    tbl_sfp.to_excel(w, sheet_name="Динамика СФП по месяцам")

print(f"Сохранено: {excel_path}")